In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [ ]:
# fetch dataset
adult = fetch_ucirepo(id=2)
adult = adult.data.features.join(adult.data.targets, how="inner")

In [ ]:
adult.head(3)

## Basic Preprocessing Steps

### 1. Drop missing values

In [ ]:
# Drop missing values
adult.dropna(inplace=True)

### 2. Copy DataFrame for posterity

In [ ]:
df = adult.copy()

In [ ]:
adult["income"].value_counts()

### 3. Encode categorical variables

In [ ]:
def outcome_merge(val):
    if val == "<=50K" or val == "<=50K.":
        return 0
    else:
        return 1

In [ ]:
df["income"] = df["income"].apply(outcome_merge)

In [ ]:
#  sex, count and percentages above_50k

income_by_sex = df.groupby("sex")["income"].agg(
    ["count", lambda x: (x.sum() / x.count()) * 100]
)
income_by_sex.columns = ["count", "percentage_above_50k"]
income_by_sex

In [ ]:
#  race, count and percentages above_50k

income_by_race = df.groupby("race")["income"].agg(
    ["count", lambda x: (x.sum() / x.count()) * 100]
)
income_by_race.columns = ["count", "percentage_above_50k"]
income_by_race

In [ ]:
df["race"] = df["race"].replace("Amer-Indian-Eskimo", "Native American or Inuit")

### 4. Split the data

In [ ]:
# Split data
X = df.drop("income", axis=1)
y = df["income"]

In [ ]:
for col in X.columns:
    if isinstance(X[col], object):
        X[col] = X[col].astype("category")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [ ]:
y_train.value_counts()

## Train XGBoost Model

In [ ]:
model = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    enable_categorical=True,
)
model.fit(X_train, y_train)

## Evaluate XGBoost Model

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(classification_report(y_test, y_pred))

# Bias and Fairness Analysis with EquiBoots

**Equiboots supports a point estimate fairness analysis on a model's operating point (e.g., optimal threshold) as well as on multiple bootstraps with replacement.**


To initialize an analysis with equiboots:

1. Define a fairness Dataframe with the variables of interest.
2. Initialize an equiboots object using:
    - Ground truth (y_true)
    - Model probabilities (y_prob)
    - Model predictions (y_pred)
3. Identify the columns/variables that we will be assessing (e.g., race, sex)

In [ ]:
fairness_df = X_test[["race", "sex"]].reset_index(drop=True)
y_test = y_test.to_numpy()

In [ ]:
import numpy as np
import equiboots as eqb

int_list = np.linspace(0, len(y_test), num=len(y_test), dtype=int).tolist()

eq2 = eqb.EquiBoots(
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob,
    fairness_df=fairness_df,
    fairness_vars=["race"],
    seeds=int_list,
    reference_groups=["White"],
    task="binary_classification",
    bootstrap_flag=True,
    num_bootstraps=5001,
    boot_sample_size=len(y_test), 
    balanced=False,  
    group_min_size=20,
    stratify_by_outcome=True,  
)

# Set seeds after initialization
eq2.set_fix_seeds(int_list)
print("seeds", eq2.seeds)

In [ ]:
eq2.grouper(groupings_vars=["race"])
boots_race_data = eq2.slicer("race")

In [ ]:
y_test

In [ ]:
race_metrics = eq2.get_metrics(boots_race_data)

In [ ]:
dispa = eq2.calculate_disparities(race_metrics, "race")

In [ ]:
eqb.eq_group_metrics_plot(
    group_metrics=dispa,
    metric_cols=[
        "Accuracy_Ratio", 
        "Precision_Ratio", 
        "Predicted_Prevalence_Ratio",
        "Prevalence_Ratio", 
        "FP_Rate_Ratio", 
        "TN_Rate_Ratio", 
        "Recall_Ratio",
    ],
    name="race",
    categories="all",
    plot_type="violinplot",
    color_by_group=True,
    strict_layout=True,
    figsize=(12, 6),
    show_grid=False,
    leg_cols=7,
    max_cols=4,
    filename="race_metrics_violin_plot.svg",
    save_path="./images",
)

In [ ]:
diffs = eq2.calculate_differences(race_metrics, "race")

In [ ]:
eqb.eq_group_metrics_plot(
    group_metrics=diffs,
    metric_cols=[
        "Accuracy_diff", "Precision_diff", "Predicted_Prevalence_diff",
        "Prevalence_diff", "FP_Rate_diff", "TN_Rate_diff", "Recall_diff",
    ],
    name="race",
    categories="all",
    plot_type="violinplot",
    color_by_group=True,
    strict_layout=True,
    figsize=(12, 6),
    leg_cols=7,
    max_cols=4,
    show_grid=False,
    filename="race_diff_violin_plot.svg",
    save_path="./images",
)

In [ ]:
metrics_boot = [
    "Accuracy_diff", "Precision_diff", "Recall_diff", "F1_Score_diff",
    "Specificity_diff", "TP_Rate_diff", "FP_Rate_diff", "FN_Rate_diff",
    "TN_Rate_diff", "Prevalence_diff", "Predicted_Prevalence_diff",
    "ROC_AUC_diff", "Average_Precision_Score_diff", "Log_Loss_diff",
    "Brier_Score_diff", "Calibration_AUC_diff"
]

test_config = {
    "test_type": "bootstrap_test",
    "alpha": 0.05,
    "adjust_method": "bonferroni",
    "confidence_level": 0.95,
    "classification_task": "binary_classification",
    "tail_type": "two_tailed",
    "metrics": metrics_boot,
}

stat_test_results = eq2.analyze_statistical_significance(
    metric_dict=race_metrics,
    var_name="race",
    test_config=test_config,
    differences=diffs,
)

In [ ]:
stat_metrics_table_diff = eqb.metrics_table(
    race_metrics,
    statistical_tests=stat_test_results,
    differences=diffs,
    reference_group="White",
)

In [ ]:
stat_metrics_table_diff

In [ ]:
import xgboost
xgboost.__version__

In [ ]:
eqb.eq_group_metrics_plot(
    group_metrics=diffs,
    metric_cols=metrics_boot,
    name="race",
    categories="all",
    figsize=(12, 12),
    plot_type="violinplot",
    color_by_group=True,
    show_grid=True,
    max_cols=4,
    strict_layout=True,
    show_pass_fail=False,
    statistical_tests=stat_test_results,
    save_path="./images",
    filename="race_diff_stat_sig_violin_plot.svg",
)

In [ ]:
eqb.eq_plot_bootstrapped_group_curves(
    boot_sliced_data=boots_race_data,
    curve_type="roc",
    title="Bootstrapped ROC Curve by Race",
    bar_every=100,
    n_bins=10,
    figsize=(6, 6),
    color_by_group=True,
    save_path="./images",
    filename="race_boots_roc_curve.svg"
)

In [ ]:
eqb.eq_plot_bootstrapped_group_curves(
    boot_sliced_data=boots_race_data,
    curve_type="roc",
    title="Bootstrapped ROC Curve by Race",
    bar_every=100,
    subplots=True,
    n_bins=10,
    n_cols=1,
    figsize=(6, 6),
    color_by_group=True,
    save_path="./images",
    filename="race_boots_roc_curve_subplots.svg"
)

In [ ]:
eqb.eq_plot_bootstrapped_group_curves(
    boot_sliced_data=boots_race_data,
    curve_type="pr",
    title="Bootstrapped PR Curve by Race",
    bar_every=100,
    subplots=True,
    n_bins=10,
    n_cols=1,
    figsize=(6, 6),
    color_by_group=True,
    save_path="./images",
    filename="race_boots_pr_curve_subplots.svg"
)

In [ ]:
eqb.eq_plot_bootstrapped_group_curves(
    boot_sliced_data=boots_race_data,
    curve_type="calibration",
    title="Bootstrapped Calibration Curve by Race",
    subplots=True,
    bar_every=10,
    n_cols=1,
    n_bins=10,
    figsize=(6, 6),
    color_by_group=True,
    save_path="./images",
    filename="race_boots_calibration_curve_subplots.svg"
)

In [ ]:
eqb.eq_plot_bootstrap_forest(
    group_boot_metrics=race_metrics,
    metric="ROC AUC",
    reference_group="White",
    title="AUROC - Bootstrapped Race Metrics",
    figsize=(8, 6),
    statistical_tests= stat_test_results,
    save_path="./images",
    filename="race_boots_auroc_forest_plot.svg"
    )

In [ ]:
eqb.calculate_bootstrap_stats(
    group_boot_metrics=race_metrics,
    metric="ROC AUC"
)
